# 01 — Exploratory Data Analysis

This notebook explores the multi-table fraud dataset to understand distributions, class balance, and inter-table relationships before modelling.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (10, 5)

## Load all tables

In [ ]:
customers = pd.read_csv("../data/synthetic/customers.csv")
accounts = pd.read_csv("../data/synthetic/accounts.csv")
devices = pd.read_csv("../data/synthetic/devices.csv")
train = pd.read_csv("../data/synthetic/train.csv")
test = pd.read_csv("../data/synthetic/test.csv")
login_events = pd.read_csv("../data/synthetic/login_events.csv")
investigations = pd.read_csv("../data/synthetic/investigations.csv")
alerts = pd.read_csv("../data/synthetic/alerts.csv")
edges = pd.read_csv("../data/synthetic/edges.csv")

for name, df in [("customers", customers), ("accounts", accounts), ("devices", devices),
                  ("train", train), ("test", test), ("login_events", login_events),
                  ("investigations", investigations), ("alerts", alerts), ("edges", edges)]:
    print(f"{name:20s}  {df.shape[0]:>8,} rows x {df.shape[1]:>3} cols")

## Transaction fraud and ATO label distributions

In [ ]:
print("=== Training set ===")
print(f"Fraud rate: {train['label_fraud'].mean():.4f}")
print(f"ATO rate:   {train['label_ato'].mean():.4f}")
print(f"\n=== Test set ===")
print(f"Fraud rate: {test['label_fraud'].mean():.4f}")
print(f"ATO rate:   {test['label_ato'].mean():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
train["label_fraud"].value_counts().sort_index().plot.bar(ax=axes[0], color=["steelblue", "salmon"])
axes[0].set_title("Fraud label distribution (train)")
axes[0].set_xlabel("label_fraud")
train["label_ato"].value_counts().sort_index().plot.bar(ax=axes[1], color=["steelblue", "salmon"])
axes[1].set_title("ATO label distribution (train)")
axes[1].set_xlabel("label_ato")
plt.tight_layout()
plt.show()

## Amount distribution by fraud label

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
train[train["label_fraud"] == 0]["amount"].clip(upper=3000).hist(bins=60, alpha=0.6, label="Non-fraud", ax=ax)
train[train["label_fraud"] == 1]["amount"].clip(upper=3000).hist(bins=60, alpha=0.6, label="Fraud", ax=ax)
ax.set_xlabel("Amount")
ax.set_title("Transaction amount distribution by fraud label")
ax.legend()
plt.show()

## Channel and merchant risk by fraud status

In [ ]:
fraud_by_channel = train.groupby("channel")["label_fraud"].mean().sort_values(ascending=False)
fraud_by_merchant = train.groupby("merchant_category")["label_fraud"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fraud_by_channel.plot.bar(ax=axes[0], color="salmon")
axes[0].set_title("Fraud rate by channel")
axes[0].set_ylabel("Fraud rate")
fraud_by_merchant.plot.bar(ax=axes[1], color="salmon")
axes[1].set_title("Fraud rate by merchant category")
axes[1].set_ylabel("Fraud rate")
plt.tight_layout()
plt.show()

## Device and geo novelty vs fraud

In [ ]:
print("Fraud rate when device is NEW vs SEEN:")
print(train.groupby("device_seen_before")["label_fraud"].mean())
print("\nFraud rate when geo is NEW vs SEEN:")
print(train.groupby("geo_seen_before")["label_fraud"].mean())

## Customer segment and age band distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
customers["customer_segment"].value_counts().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Customer segment distribution")
customers["age_band"].value_counts().plot.bar(ax=axes[1], color="steelblue")
axes[1].set_title("Age band distribution")
plt.tight_layout()
plt.show()

## Investigation outcomes

In [ ]:
print(investigations["review_outcome"].value_counts())
print(f"\nMean confirmed loss (substantiated): ${investigations[investigations['review_outcome']=='substantiated']['confirmed_loss'].mean():.2f}")
print(f"Mean recovery (substantiated): ${investigations[investigations['review_outcome']=='substantiated']['recovery_amount'].mean():.2f}")

## Edge / graph summary

In [ ]:
print(f"Total edges: {len(edges):,}")
print(f"\nEdge type distribution:")
print(edges["edge_type"].value_counts())

## Key takeaways

- Fraud rate is approximately 3.5%, creating realistic class imbalance.
- New device and new geo are strong fraud risk signals.
- Crypto and cash transfer merchant categories show elevated fraud rates.
- The dataset has a realistic multi-table structure with customers, accounts, devices, transactions, logins, alerts, investigations, and entity edges.